# Efficient Calibration of the Heston Stochastic Volatility Model using Numerical Optimization

## 3. Calibration Problem Formulation

Calibrating the Heston model means choosing the five model parameters that make the theoretical option surface look as much like the observed market surface as possible. 
Because Heston has no closed-form inverse (unlike Black–Scholes, where each option uniquely implies one volatility), this recovery step has to be posed as a numerical optimization.

### 3.1 Optimization statement

Let the Heston parameter vector be

$$\theta = (v_0,\ \kappa,\ \theta_{\infty},\ \eta,\ \rho),$$

where $v_0$ is the initial variance, $\kappa$ the mean-reversion speed, $\theta_\infty$ the long-run variance, $\eta$ the vol-of-vol, and $\rho$ the spot–variance correlation. 
For a fixed valuation date, the grid of quoted tenors and strikes defines a surface of market prices $\{C^{\text{mkt}}_{i,j},\ P^{\text{mkt}}_{i,j}\}$ for $i \in \text{tenors}$, $j \in \text{strikes}$. 
The calibration problem is

$$\hat{\theta} \;=\; \arg\min_{\theta \,\in\, \Theta} \; \mathcal{L}(\theta),$$

subject to the parameter box $\Theta$ that enforces economic validity ($v_0, \theta_\infty > 0$, $\kappa, \eta > 0$, $\rho \in (-1, 1)$).

### 3.2 Objective function

The implementation uses mean squared error across the full surface, counting calls and puts as separate observations:

$$\mathcal{L}(\theta) \;=\; \frac{1}{2\,N_T N_K} \sum_{i=1}^{N_T} \sum_{j=1}^{N_K} \Big[\big(C^{\text{mkt}}_{i,j} - C^{\text{Heston}}_{i,j}(\theta)\big)^2 + \big(P^{\text{mkt}}_{i,j} - P^{\text{Heston}}_{i,j}(\theta)\big)^2\Big].$$

In code (`calculate_mse_entire`), this is the sum of squared call and put errors divided by the total point count (`count += 2` per strike × tenor cell). 
Squared error is chosen because it is smooth and differentiable — both needed by L-BFGS-B — and because it penalises large mispricings disproportionately, which matters more for hedging risk than uniform small errors.

Alternative objective functions worth noting:
- **Weighted MSE** using bid–ask spread as the per-point weight, to downweight illiquid deep wings.
- **Relative error** ($|C^{\text{mkt}} - C^{\text{Heston}}| / C^{\text{mkt}}$), which prevents expensive ATM options from dominating.
- **Implied-vol MSE**, which matches the surface in vol space rather than price space. More common in practice but adds a root-finding step per evaluation.

### 3.3 Inputs

Firstly, we obtained daily implied volatility surfaces data on SPX option from January 2012 to 2025 October from Bloomberg Terminal. These implied volatilities are computed such that the BSM option price matches with the historical option market price. Strikes are quoted as a percentage of spot.



| Input | Source | 
|---|---|
| Spot $S_0$ | `Underlying` sheet, `Mid` column |
| Risk-free rate $r$ | `Other` sheet, `Interest` column | 
| Dividend yield $q$ | `Other` sheet, `Dividend` column | 
| Strikes | `VALID_STRIKE` = `[30, 40, 60, 80, 90, 95, 97.5, 100, 105, 110, 120, 130, 150, 300]` (% of spot) | 
| Tenors | `VALID_TENOR` = `1W`–`2Y`, 11 expiries | 
| Market prices | `blackScholes(...)` applied to quoted implied vols | 




### 3.4 Parameter space and initial guess

We selected the 1M ATM Volatility to be initial guess for inital variance in the Heston Model. We initialized vov as 0.5 and spot/vol correlation as -0.5

In [ ]:

initial_guess = [atm_vol**2,  2.0,  atm_vol**2,  0.5,  -0.5]

bounds = [
    (1e-6, 2.0),     # v0
    (1e-4, 20.0),    # kappa
    (1e-6, 2.0),     # theta
    (1e-4, 5.0),     # eta
    (-0.999, 0.999), # rho
]

### 3.5 Optimization

The code uses `scipy.optimize.minimize` with **L-BFGS-B**, a quasi-Newton gradient method with box constraints. 
The choice trades global coverage for speed — L-BFGS-B is fast but local. 
Robustness is handled through the initial guess (seeded from market ATM vol so it starts inside the right basin) and through guarded return values inside `objective(x)`:

In [ ]:
if not np.isfinite(heston_call) or not np.isfinite(heston_put):
    return 1e12

so that numerically unstable parameter combinations don't crash the optimizer but look unattractive to it instead.

### 3.6 Challenges

#### Non-linearity

$C^{\text{Heston}}(\theta)$ depends on $\theta$ through a characteristic function and an inverse Fourier transform (`pf.HestonFft`). 
This is smooth but highly non-linear: $\kappa$ and $\theta_\infty$ enter exponentially through the CIR variance mean, $\rho$ enters as a cross-term in the drift of the log-price, and $\eta$ multiplies a Brownian increment inside the variance SDE. 
The loss surface $\mathcal{L}(\theta)$ inherits all of this — no convexity guarantees.

#### Multiple local minima

The Heston calibration problem is *well-documented* to be non-convex. 
The classic symptom is the $\kappa$–$\theta_\infty$–$\eta$ trade-off: several combinations produce nearly identical long-dated smiles, differing only in how they extrapolate to tenors outside the calibration grid. 
The Feller condition $2\kappa\theta_\infty \ge \eta^2$ carves the parameter space further, and near-equivalent minima can straddle it.

Practical mitigations (not all implemented here):
- **Multi-start** — run L-BFGS-B from several initial points and keep the best minimum.
- **Global-then-local** — differential evolution or CMA-ES first to get into a good basin, L-BFGS-B to polish.
- **Regularization** — add a small penalty toward economically reasonable parameters to break near-flat directions in the loss.

#### Sensitivity to initial guesses

Because the solver is local, the answer depends on `initial_guess`. 
The current seeding ($v_0 = \theta_\infty = \sigma_{\text{ATM,1M}}^2$, $\kappa=2$, $\eta=0.5$, $\rho=-0.5$) is a reasonable equity default, but there is no guarantee this is the basin of the global optimum on a given date. 
A sensitivity check worth running: perturb the initial guess by ±30% on each component, re-calibrate, and record the spread in final parameters. 
Small final-parameter dispersion ⇒ well-identified calibration; large dispersion ⇒ surface doesn't pin down all five parameters on this date and regularization is needed.